In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import max_error
import pandas as pd
import os
import joblib
from itertools import compress

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints
from neuro_bes.preprocessing import profile_transform
from neuro_bes.postprocessing import evaluation

In [ ]:
tf.config.threading.set_intra_op_parallelism_threads(50)
tf.config.threading.set_inter_op_parallelism_threads(50)

In [ ]:
batch_test_full_shot=[]
file_path="/home/molnarbalazs/data/BES_ML_modelling/W7X_experimental_data_from_flap"    
#get all h5 files from "/home/molnarbalazs/data/BES_ML_modelling/W7X_experimental_data_from_flap" folder which are under a MB in size
for file in sorted(os.listdir(file_path), key=str.lower):
    if file.endswith(".h5") and os.path.getsize(os.path.join(file_path, file)) < 1e6:
        batch_test_full_shot.append(besInferenceDatapoints(path=os.path.join(file_path, file)))

In [ ]:
len(batch_test_full_shot)

In [ ]:
pipeline=joblib.load("preprocessing_pipeline_newcuration.joblib")
#edit the id field ofpipeline['beam_int_scaler'].integral_emissions_ by adding the postfix "_train"
for i in pipeline['beam_int_scaler'].integral_emissions_:
    i['id']=i['id']+"_train"
model=tf.keras.models.load_model("density_prediction_model_newcuration.keras")
pipeline_batch_test_full_shot=pipeline.transform(batch_test_full_shot)
for batch in pipeline_batch_test_full_shot:
    y_pred_scaled = model.predict(batch.emissions)
    y_pred_scaled=y_pred_scaled.reshape(y_pred_scaled.shape[0], -1)
    batch.densities=y_pred_scaled
batch_test_pred_full_shot=pipeline.inverse_transform(pipeline_batch_test_full_shot)

In [ ]:
for bes_data_to_plot in batch_test_pred_full_shot[1:]:
    # plot the density and prediction profiles on a heatmap with same color scale
    # set seismic colormap with white at 0 and red/blue for positive/negative values
    bes_data_to_plot.export_to_h5(path_to_dir="/data2/W7-X/processed_data/APDCAM/NN_recon")
    shot=bes_data_to_plot.ID[3:].replace(".", "")
    time_instances=[float(i[14:19]) for i in bes_data_to_plot.tags]
    r_coord=bes_data_to_plot.grid
    downsampling=1
    plt.figure(figsize=(12,6))
    plt.subplot(2,1,1)
    # colorbar with same scale for both plots
    im = plt.pcolormesh(time_instances[::downsampling], r_coord, bes_data_to_plot.emissions[::downsampling].T, cmap='RdBu_r')
    cbar = plt.colorbar(im)
    cbar.ax.set_ylabel('mV',fontsize=18, labelpad=10)
    cbar.ax.tick_params(labelsize=12)
    vmax = np.max(bes_data_to_plot.emissions)
    im.set_clim(0, vmax)
    plt.title('Light Profiles '+bes_data_to_plot.ID,fontsize=20)
    plt.xlabel('Time',fontsize=18, labelpad=10)
    plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.subplot(2,1,2)
    im2 = plt.pcolormesh(time_instances[::downsampling],r_coord, bes_data_to_plot.densities[::downsampling].T*1e19, cmap='RdBu_r')
    cbar = plt.colorbar(im2)
    cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
    cbar.ax.tick_params(labelsize=16)
    vmax = np.max(bes_data_to_plot.densities*1e19)
    im2.set_clim(0, vmax)
    plt.title('Predicted Density Profiles '+bes_data_to_plot.ID,fontsize=20)
    plt.xlabel('Time (s)',fontsize=18, labelpad=10)
    plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.savefig("/data2/W7-X/processed_data/APDCAM/NN_recon/time_evolution_shot_"+shot+".png", bbox_inches='tight')
    plt.show()


In [ ]:
#read h5 file with h5py and print the keys

import h5py
with h5py.File(os.path.join("/data2/W7-X/processed_data/APDCAM/NN_recon/","Dataset_Na_0_we_20241008.046.h5"), 'r') as f:
    print(f.keys())
    pred=f['density'][()]
plt.plot(pred.T)